In [1]:
import pyspark
import os
from pyspark.sql import SparkSession

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("modules-6-assignments") \
        .getOrCreate()

# Question 1: Install Spark and PySpark
## Execute spark.version. What's the output?
spark.version

In [ ]:
# Get Taxi Data 
if not os.path.exists("yellow_tripdata_2025-11.parquet"):
    !wget wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [ ]:
df = spark.read.option("header","true") \
    .parquet("yellow_tripdata_2025-11.parquet")
df.createOrReplaceTempView("YellowTripData202511")
df.printSchema()
df.show(5)

In [ ]:
#Question 2: Yellow November 2025
# Repartition the Dataframe to 4 partitions and save it to parquet.
# What is the average size of the Parquet files?
df = df.repartition(4)
df.write.parquet("yellow_tripdata_2025-11_repartitioned", mode="overwrite")
!ls -lh yellow_tripdata_2025-11_repartitioned/

In [ ]:
#Question 3: Count records
spark.sql("""
    SELECT COUNT(*) as trips 
    FROM YellowTripData202511 
    WHERE tpep_pickup_datetime >= '2025-11-15 00:00:00'
    AND tpep_pickup_datetime < '2025-11-16 00:00:00'
""").show()

In [ ]:
# Question 4: Longest trip
spark.sql("""
    SELECT ROUND((DATEDIFF(second, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600),1) as trip_time_hours
    FROM YellowTripData202511
    ORDER BY trip_time_hours DESC
    LIMIT 1
    """).show()

In [ ]:
# Question 6: Least frequent pickup location zone
if not os.path.exists("taxi_zone_lookup.csv"):
    !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [ ]:
df_zones = spark.read.option("header","true") \
           .csv("taxi_zone_lookup.csv")
df_zones.createOrReplaceTempView("ZoneLookup")

In [ ]:
  spark.sql("""                                                                                                                      
      SELECT zlu.Zone, COUNT(*) as trips                                                                                             
      FROM YellowTripData202511 ytd                                                                                                  
      JOIN ZoneLookup zlu ON zlu.LocationID = ytd.PULocationID
      GROUP BY zlu.Zone
      ORDER BY trips ASC
      LIMIT 5
  """).show(truncate=False)